# 04: Update compression on BloodMNIST (alpha 0.5)

Runs FedAvg with int8 quantization and top-k sparsification (keep 10% and 1%, with error feedback) and logs accuracy and bytes sent per round. Same settings as notebook 01. FedAvg with no compression is in `results/step4_bloodmnist_3seeds.json`. Turn on a T4 GPU first. About 10 minutes.

In [ ]:
!nvidia-smi -L
!rm -rf /content/fedsurveil && git clone -q https://github.com/Nathaniel47/fedsurveil /content/fedsurveil
%cd /content/fedsurveil
!pip install -q -r requirements-ml.txt 2>&1 | tail -1
!python -m fedsurveil.data.fetch
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.__version__, torch.cuda.is_available())

In [ ]:
import subprocess, sys
BASE = ["--dataset", "bloodmnist", "--rounds", "30", "--lr", "0.02", "--momentum", "0.9", "--local-epochs", "2", "--alpha", "0.5"]
for seed in [0, 1, 2]:
    for extra in (["--compress", "int8"], ["--compress", "topk", "--topk-fraction", "0.1"], ["--compress", "topk", "--topk-fraction", "0.01"]):
        cmd = [sys.executable, "-m", "fedsurveil.fl.simulate", *BASE, "--seed", str(seed), *extra]
        subprocess.run(cmd, check=True, capture_output=True)
        print("done", seed, extra)

In [ ]:
import glob, json, os
# Compact copy: [round, acc, macro_f1, worst_site_acc, mean_site_acc] per round, and bytes sent per round
for p in sorted(glob.glob("runs/*.jsonl")):
    rows = [json.loads(line) for line in open(p)]
    rounds = [r for r in rows if r["type"] == "round"]
    curve = [[r["round"], round(r["global_acc"], 4), round(r["macro_f1"], 4), round(r["site_acc_spread"]["min"], 4), round(r["site_acc_spread"]["mean"], 4)] for r in rounds]
    print("RUN", os.path.basename(p), json.dumps(curve), "bytes_up_per_round", rounds[0]["bytes_up"], "bytes_down_per_round", rounds[0]["bytes_down"])